
# 1. The Data

This notebook contains all the data used for our November 2 report, including the EDGAR filings, CRSP matching, earnings release texts, and three independent sources of release timestamps.

The notebook can be run from top to bottom. The data acquisition scripts are saved to disk by the cells containing them, so they can be reviewed and executed separately. However, they are disabled by default and only run if `RUN_FROM_SCRATCH` is enabled, since they require WRDS access and approximately one day to download the data.



In [ ]:
#setup
!pip install -q "pandas>=2.0"

import glob, os, zipfile
import pandas as pd

#the acquisition pass is off. It needs a WRDS account and about a day of fetching from EDGAR, and it is what produced the files in nov2_data.
RUN_FROM_SCRATCH = False
WRDS_USER = ""

print("pandas", pd.__version__)

pandas 2.2.3


In [ ]:
#data

import os
from google.colab import drive

drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/S08 NLP/1_2nov/nov2_data"

NEED = [
    "attrition_funnel.csv",
    "raw_counts.csv",
    "matched_sample.csv",
    "extraction_report.csv",
    "timing_validation.csv",
    "wire_times.csv",
    "three_source_timing.csv",
    "event_table.csv"
]

#we check data files

if not os.path.isdir(DATA):
    raise FileNotFoundError(f"Folder not found: {DATA}")

missing = []

for f in NEED:
    path = os.path.join(DATA, f)
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        missing.append(f)

if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

RESULTS = "/content/drive/MyDrive/S08 NLP/results/1_results"
MANUAL = "/content/drive/MyDrive/S08 NLP/2_EDGAR text/Manual validation"

os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MANUAL, exist_ok=True)

for f in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, f)
    if os.path.isfile(path):
        print(f"{os.path.getsize(path):>12,}  {f}")

print("\nresults ->", RESULTS)
print("manual  ->", MANUAL)

Mounted at /content/drive
      37,597  acceptance_header_check.csv
         311  attrition_funnel.csv
  92,679,435  event_table.csv
   1,874,799  extraction_report.csv
   2,983,458  matched_sample.csv
         550  raw_counts.csv
  12,312,462  three_source_timing.csv
   8,382,896  timing_validation.csv
  12,012,726  wire_times.csv

results -> /content/drive/MyDrive/S08 NLP/results/1_results
manual  -> /content/drive/MyDrive/S08 NLP/results/2_EDGAR text/Manual validation


In [ ]:
%%writefile funnel_counts.py
#we count all 8-K Item 2.02 filings using the SEC bulk submissions archive.
#we process the ZIP without extracting it and save the raw counts to CSV.
#this is the first step of the attrition funnel, before CRSP matching and filtering.

from __future__ import annotations

import argparse
import csv
import io
import json
import os
import sys
import urllib.request
import zipfile
from collections import Counter, defaultdict

BULK_URL = "https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip"
USER_AGENT = os.environ.get("SEC_USER_AGENT", "MIT 15.S08 project mbonnetderoovere@gmail.com")

START_YEAR = 2006
END_YEAR = 2025


def download(url: str, dest: str) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", 0))
        done = 0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                pct = 100 * done / total
                print(f"\r  {done/1e9:.2f} / {total/1e9:.2f} GB  ({pct:.1f}%)",
                      end="", file=sys.stderr, flush=True)
    print(file=sys.stderr)


def iter_filing_tables(zf: zipfile.ZipFile):
    for info in zf.infolist():
        name = info.filename
        if not name.endswith(".json") or not name.startswith("CIK"):
            continue
        try:
            with zf.open(info) as fh:
                doc = json.load(fh)
        except (json.JSONDecodeError, zipfile.BadZipFile, OSError):
            yield None, name
            continue

        if isinstance(doc, dict) and "filings" in doc:
            recent = doc.get("filings", {}).get("recent")
            if recent:
                yield recent, name
        elif isinstance(doc, dict) and "form" in doc:
            yield doc, name


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip", required=True, help="path to submissions.zip")
    ap.add_argument("--out", default="raw_counts.csv")
    args = ap.parse_args()

    if not os.path.exists(args.zip):
        print(f"downloading {BULK_URL}", file=sys.stderr)
        download(BULK_URL, args.zip)

    by_year = Counter()            #item 2.02 filings per year
    all_8k_by_year = Counter()     #every 8-K, for context
    filers_by_year = defaultdict(set)
    unreadable = 0
    members = 0

    with zipfile.ZipFile(args.zip) as zf:
        for table, name in iter_filing_tables(zf):
            members += 1
            if table is None:
                unreadable += 1
                continue

            forms = table.get("form", [])
            items = table.get("items", [])
            dates = table.get("filingDate", [])
            cik = name.split("-")[0][3:].removesuffix(".json").lstrip("0")

            for i, form in enumerate(forms):
                if form != "8-K":
                    continue
                date = dates[i] if i < len(dates) else ""
                if len(date) < 4:
                    continue
                year = int(date[:4])
                if not (START_YEAR <= year <= END_YEAR):
                    continue
                all_8k_by_year[year] += 1

                codes = (items[i] if i < len(items) else "").split(",")
                if "2.02" in [c.strip() for c in codes]:
                    by_year[year] += 1
                    filers_by_year[year].add(cik)

            if members % 20000 == 0:
                print(f"\r  {members:,} filers read", end="",
                      file=sys.stderr, flush=True)

    print(file=sys.stderr)

    with open(args.out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["year", "item_2_02_filings", "all_8k_filings",
                    "distinct_filers_with_2_02"])
        for year in range(START_YEAR, END_YEAR + 1):
            w.writerow([year, by_year[year], all_8k_by_year[year],
                        len(filers_by_year[year])])
        w.writerow(["TOTAL", sum(by_year.values()), sum(all_8k_by_year.values()), ""])

    print(f"filers read      : {members:,}")
    print(f"unreadable members: {unreadable:,}")
    print(f"Item 2.02 total  : {sum(by_year.values()):,}")
    print(f"all 8-K total    : {sum(all_8k_by_year.values()):,}")
    print(f"written          : {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing funnel_counts.py


In [ ]:
%%writefile build_event_table.py
#we build an event table for all 8-K Item 2.02 filings from 2006 to 2025 using SEC data.
#we record filing identifiers, acceptance timestamps in New York time, and assign each
#filing to the correct trading session. We download release texts only after CRSP matching.

#Usage: python build_event_table.py --zip <submissions.zip> --out event_table.csv

from __future__ import annotations

import argparse
import csv
import json
import os
import sys
import zipfile
from datetime import datetime, timedelta, time
from zoneinfo import ZoneInfo

UTC = ZoneInfo("UTC")
ET = ZoneInfo("America/New_York")

START_YEAR = 2006
END_YEAR = 2025

CLOSE = time(16, 0)
OPEN = time(9, 30)


def parse_acceptance(raw: str):
    if not raw:
        return None
    try:
        dt = datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=UTC)
    return dt.astimezone(ET)


def load_header_times(path: str) -> dict[str, datetime]:
    #we extract ACCEPTANCE-DATETIME for each filing from its header using fetch_acceptance.py.
    #the timestamp is recorded in Eastern time (YYYYMMDDHHMMSS), without a timezone offset.
    if not path or not os.path.exists(path):
        return {}
    out: dict[str, datetime] = {}
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            s = (r.get("acceptance_header_et") or "").strip()
            if len(s) != 14 or not s.isdigit():
                continue
            out[r["accession"]] = datetime.strptime(
                s, "%Y%m%d%H%M%S").replace(tzinfo=ET)
    return out


def session_for(acc_et: datetime | None, filing_date: str):
    #we determine when each filing becomes public and assign it to a session date.
    #we use the acceptance time by default and move after-close releases to the next
    #calendar day, then to the next trading day in wrds_match_and_funnel.py.
    if acc_et is None:
        return None, filing_date, "unknown", "no_acceptance_time"

    acc_date = acc_et.date().isoformat()
    if filing_date and filing_date > acc_date:
        # EDGAR took the filing but disseminated it the following business morning.
        #the report says we treat that as public at 06:00 ET on
        #the filing date, so the timestamp has to say 06:00 and not the
        #acceptance time, which is the previous evening and would date the release before anybody could see it.
        y, m, d = (int(x) for x in filing_date.split("-"))
        return (datetime(y, m, d, 6, 0, tzinfo=ET), filing_date,
                "pre_open", "filed_after_acceptance")

    t = acc_et.timetz().replace(tzinfo=None)
    if t >= CLOSE:
        nxt = (acc_et + timedelta(days=1)).date().isoformat()
        return acc_et, nxt, "after_close", ""
    if t < OPEN:
        return acc_et, acc_date, "pre_open", ""
    return acc_et, acc_date, "intraday", ""


def iter_tables(zf: zipfile.ZipFile):
    for info in zf.infolist():
        name = info.filename
        if not name.endswith(".json") or not name.startswith("CIK"):
            continue
        try:
            with zf.open(info) as fh:
                doc = json.load(fh)
        except Exception:
            continue
        if isinstance(doc, dict) and "filings" in doc:
            t = doc.get("filings", {}).get("recent")
            if t:
                yield t, name, doc.get("name", "")
        elif isinstance(doc, dict) and "form" in doc:
            yield doc, name, ""


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip", required=True)
    ap.add_argument("--out", default="event_table.csv")
    ap.add_argument("--acceptance", default="acceptance_times.csv",
                    help="ACCEPTANCE-DATETIME per accession, from "
                         "fetch_acceptance.py. Used in preference to "
                         "the submissions JSON wherever it has a row.")
    args = ap.parse_args()

    header_et = load_header_times(args.acceptance)
    if header_et:
        print(f"header acceptance times: {len(header_et):,}", flush=True)
    else:
        print("no header file; falling back to the submissions JSON, "
              "which is wrong for about two filings in three",
              file=sys.stderr)

    rows = 0
    src = {"header": 0, "json": 0}
    flags = {"": 0, "no_acceptance_time": 0, "filed_after_acceptance": 0}
    when = {"after_close": 0, "pre_open": 0, "intraday": 0, "unknown": 0}

    with zipfile.ZipFile(args.zip) as zf, \
         open(args.out, "w", newline="", encoding="utf-8") as out:
        w = csv.writer(out)
        w.writerow([
            "cik", "company", "accession", "filing_date", "report_date",
            "acceptance_utc", "acceptance_et", "acceptance_source",
            "release_et", "session_date", "when",
            "timing_flag", "items", "primary_document", "filing_dir_url",
        ])

        for table, member, company in iter_tables(zf):
            cik_raw = member.split("-")[0][3:].removesuffix(".json")
            cik = cik_raw.lstrip("0")
            forms = table.get("form", [])
            items = table.get("items", [])
            fdates = table.get("filingDate", [])
            rdates = table.get("reportDate", [])
            accs = table.get("accessionNumber", [])
            accept = table.get("acceptanceDateTime", [])
            prim = table.get("primaryDocument", [])

            for i, form in enumerate(forms):
                if form != "8-K":
                    continue
                fd = fdates[i] if i < len(fdates) else ""
                if len(fd) < 4:
                    continue
                year = int(fd[:4])
                if not (START_YEAR <= year <= END_YEAR):
                    continue
                codes = [c.strip() for c in
                         (items[i] if i < len(items) else "").split(",")]
                if "2.02" not in codes:
                    continue

                acc = accs[i] if i < len(accs) else ""
                raw_acc_dt = accept[i] if i < len(accept) else ""
                acc_et = header_et.get(acc)
                if acc_et is not None:
                    src["header"] += 1
                else:
                    acc_et = parse_acceptance(raw_acc_dt)
                    src["json"] += 1
                rel_et, sess, wh, flag = session_for(acc_et, fd)

                nodash = acc.replace("-", "")
                url = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
                       f"{nodash}/")

                w.writerow([
                    cik, company, acc, fd,
                    rdates[i] if i < len(rdates) else "",
                    raw_acc_dt,
                    acc_et.isoformat() if acc_et else "",
                    "header" if acc in header_et else "json",
                    rel_et.isoformat() if rel_et else "",
                    sess, wh, flag,
                    items[i] if i < len(items) else "",
                    prim[i] if i < len(prim) else "",
                    url,
                ])
                rows += 1
                flags[flag] = flags.get(flag, 0) + 1
                when[wh] = when.get(wh, 0) + 1

    print(f"rows written            : {rows:,}")
    print(f"  from the filing header: {src['header']:,}")
    print(f"  from the JSON         : {src['json']:,}")
    print(f"  after close           : {when['after_close']:,}")
    print(f"  pre-open              : {when['pre_open']:,}")
    print(f"  intraday              : {when['intraday']:,}")
    print(f"  unknown time          : {when['unknown']:,}")
    print(f"flagged filed>acceptance: {flags['filed_after_acceptance']:,}")
    print(f"flagged no timestamp    : {flags['no_acceptance_time']:,}")
    print(f"written                 : {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing build_event_table.py


In [ ]:
%%writefile fetch_acceptance.py
#we extract ACCEPTANCE-DATETIME directly from each filing header, as the submissions
#JSON is unreliable (201 mismatches out of 300 filings: 128 mislabeled as UTC and
#73 with an incorrect offset). We fetch one header per filing, resume from saved
#results, and save the timestamps in a CSV keyed by accession for build_event_table.py.
#python fetch_acceptance.py --events event_table.csv

from __future__ import annotations

import argparse
import csv
import os
import re
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

USER_AGENT = "MIT 15.S08 project mbonnetderoovere@gmail.com"
RATE = 9.0
WORKERS = 8
HEADER_RE = re.compile(r"<ACCEPTANCE-DATETIME>\s*(\d{14})")


class RateLimiter:
    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch_head(url: str, limiter: RateLimiter, retries: int = 3) -> str | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                #the header is the first thing in the submission file.
                return r.read(4096).decode("utf-8", "replace")
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def already_done(path: str) -> set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, newline="", encoding="utf-8") as f:
        return {r["accession"] for r in csv.DictReader(f)}


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default="event_table.csv")
    ap.add_argument("--out", default="acceptance_times.csv")
    ap.add_argument("--workers", type=int, default=WORKERS)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    seen, rows = set(), []
    with open(args.events, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            a = r["accession"]
            if a and a not in seen:
                seen.add(a)
                rows.append((a, r["cik"]))

    done = already_done(args.out)
    fresh = not done
    if done:
        before = len(rows)
        rows = [r for r in rows if r[0] not in done]
        print(f"resuming: {before - len(rows):,} stored, {len(rows):,} to go",
              flush=True)
    if args.limit:
        rows = rows[:args.limit]
    if not rows:
        print("nothing to do")
        return 0

    limiter = RateLimiter(RATE)
    lock = threading.Lock()
    t0 = time.monotonic()
    n = {"done": 0, "missing": 0}

    out = open(args.out, "a", newline="", encoding="utf-8")
    w = csv.writer(out)
    if fresh:
        w.writerow(["accession", "acceptance_header_et"])

    def work(item) -> None:
        acc, cik = item
        url = (f"https://www.sec.gov/Archives/edgar/data/{str(cik).lstrip('0')}/"
               f"{acc.replace('-', '')}/{acc}.txt")
        head = fetch_head(url, limiter)
        m = HEADER_RE.search(head or "")
        stamp = m.group(1) if m else ""
        with lock:
            w.writerow([acc, stamp])
            n["done"] += 1
            if not stamp:
                n["missing"] += 1
            k = n["done"]
            if k % 2000 == 0:
                out.flush()
                el = time.monotonic() - t0
                rate = k / el
                left = (len(rows) - k) / rate / 3600 if rate else 0
                print(f"  {k:,}/{len(rows):,}  {rate:.1f}/s  "
                      f"{left:.1f} h left  {n['missing']:,} without a header",
                      flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, rows))
    out.close()

    print(f"\n{n['done']:,} fetched in {(time.monotonic() - t0) / 3600:.2f} h")
    print(f"{n['missing']:,} with no ACCEPTANCE-DATETIME in the header")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing fetch_acceptance.py


In [ ]:
%%writefile wrds_match_and_funnel.py
#we complete the attrition funnel by applying price, size, and analyst-coverage filters.
#we start with 383,153 filings, of which 328,317 are matched to a PERMNO.
#we process CRSP data year by year using SQL to avoid memory issues.
#WRDS authentication must be set up separately before running the script.
#python wrds_match_and_funnel.py

from __future__ import annotations

import argparse
import sys

import pandas as pd

try:
    import wrds
except ImportError:
    sys.exit("pip install wrds")


#local step: events joined to PERMNO, no WRDS needed


def matched_events(events_csv: str, link_csv: str) -> tuple[pd.DataFrame, int]:
    ev = pd.read_csv(events_csv, dtype={"cik": str}, usecols=["cik", "accession", "filing_date", "session_date", "when", "timing_flag"],)
    ev["session_date"] = pd.to_datetime(ev["session_date"])
    ev["cik"] = ev["cik"].str.lstrip("0")
    n_raw = len(ev)

    lk = pd.read_csv(link_csv, dtype=str)
    lk = lk[lk["LINKPRIM"].isin(["P", "C"])].dropna(subset=["cik"])
    lk["cik"] = lk["cik"].str.lstrip("0")
    lk["LINKDT"] = pd.to_datetime(lk["LINKDT"], errors="coerce")

    lk["LINKENDDT"] = pd.to_datetime(
        lk["LINKENDDT"].replace("E", "2099-12-31"), errors="coerce"
    ).fillna(pd.Timestamp("2099-12-31"))
    lk = lk.dropna(subset=["LINKDT"])

    j = ev.merge(lk[["cik", "gvkey", "LPERMNO", "LINKDT", "LINKENDDT"]], on="cik", how="inner")
    j = j[(j.session_date >= j.LINKDT) & (j.session_date <= j.LINKENDDT)]
    j = j.sort_values("LINKDT").drop_duplicates("accession", keep="first")
    j = j.rename(columns={"LPERMNO": "permno"})
    j["permno"] = pd.to_numeric(j["permno"], errors="coerce").astype("Int64")
    j = j.dropna(subset=["permno"])
    return j.reset_index(drop=True), n_raw


#WRDS steps, one year at a time


#the CIZ file crsp.dsf_v2 runs to 2025 where the legacy crsp.dsf stops at 2024-12-31,
#so it is the only one that covers the full holdout. The CIZ equivalent of the old shrcd 10/11 and exchcd 1/2/3 screen is spelled out in
#CIZ_COMMON below.
CIZ_COMMON = """
      sharetype = 'NS'
  and securitytype = 'EQTY'
  and securitysubtype = 'COM'
  and usincflg = 'Y'
  and issuertype in ('ACOR', 'CORP')
"""


def nyse_breakpoints(db, years: list[int]) -> pd.DataFrame:
    q = f"""
        select extract(year from dlycaldt)::int as year,
               percentile_cont(0.20) within group (order by dlycap) as nyse_p20
        from crsp.dsf_v2
        where {CIZ_COMMON}
          and primaryexch = 'N'
          and dlycap is not null
          and extract(year from dlycaldt) between {min(years)} and {max(years)}
        group by 1
        order by 1
    """
    return db.raw_sql(q)


def trading_calendar(db, years: list[int]) -> pd.DatetimeIndex:
    #we retrieve all US equity trading dates over the sample period to correctly assign
    #after-close releases to the next trading session. Without this adjustment, 5,869
    #events (1.8% of the matched sample), mainly on Fridays and before holidays, would
    #be incorrectly excluded. We retrieve the calendar with a single query.
    q = f"""
        select distinct dlycaldt as date
        from crsp.dsf_v2
        where dlycaldt between date '{min(years) - 1}-11-01'
                           and date '{max(years)}-12-31'
        order by 1
    """
    return pd.DatetimeIndex(db.raw_sql(q, date_cols=["date"])["date"])


def roll_to_session(dates: pd.Series, cal: pd.DatetimeIndex) -> pd.Series:
    #we assign each release to the first trading session on or after its date.
    #we always move forward to avoid assigning returns before the news was public.
    i = cal.searchsorted(pd.DatetimeIndex(dates), side="left")
    out = pd.Series(pd.NaT, index=dates.index, dtype="datetime64[ns]")
    ok = i < len(cal)
    out[ok] = cal[i[ok]]
    return out


def crsp_for_year(db, year: int, permnos: list[int]) -> pd.DataFrame:
    lst = ",".join(str(p) for p in permnos)
    q = f"""
        select permno, dlycaldt as date,
               abs(dlyprc) as prc, dlyprc as prc_signed, dlycap as mktcap
        from crsp.dsf_v2
        where {CIZ_COMMON}
          and primaryexch in ('N', 'A', 'Q')
          and permno in ({lst})
          and dlycaldt between date '{year - 1}-11-01' and date '{year}-12-31'
    """
    return db.raw_sql(q, date_cols=["date"])


def ibes_crsp_bridge(db) -> pd.DataFrame:
    #we use the WRDS IBES ticker-to-PERMNO link with its validity dates, keeping reliable
    #matches (scores 1–3). This replaces the slower CUSIP-based matching approach.
    q = """
        select ticker, permno, sdate, edate
        from wrdsapps.ibcrsphist
        where score <= 3
    """
    return db.raw_sql(q, date_cols=["sdate", "edate"])


def analyst_estimates(db, year: int, tickers: list[str]) -> pd.DataFrame:
    lst = ",".join("'" + t.replace("'", "''") + "'" for t in tickers)
    q = f"""
        select distinct d.ticker, d.anndats, d.analys
        from ibes.detu_epsus as d
        where d.measure = 'EPS'
          and d.fpi = '6'
          and d.analys is not null
          and d.ticker in ({lst})
          and d.anndats between date '{year - 1}-09-01' and date '{year}-12-31'
    """
    return db.raw_sql(q, date_cols=["anndats"])



class Funnel:
    def __init__(self, n0: int):
        self.rows = [("raw: 8-K Item 2.02 filings 2006-2025", n0, 0, 0.0)]
        self.n = n0

    def step(self, name: str, n_after: int) -> None:
        lost = self.n - n_after
        pct = round(100.0 * lost / self.n, 2) if self.n else 0.0
        self.rows.append((name, n_after, lost, pct))
        self.n = n_after

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows,
                            columns=["step", "remaining", "removed", "pct_removed"])


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default="event_table.csv")
    ap.add_argument("--link", default="ccm_link.csv")
    ap.add_argument("--out-sample", default="matched_sample.csv")
    ap.add_argument("--out-funnel", default="attrition_funnel.csv")
    ap.add_argument("--min-analysts", type=int, default=3)
    ap.add_argument("--min-price", type=float, default=5.0)
    ap.add_argument("--user", default=None,
                    help="WRDS username. The password is never passed here; "
                         "it is read from the local pgpass file.")
    args = ap.parse_args()

    ev, n_raw = matched_events(args.events, args.link)
    funnel = Funnel(n_raw)
    funnel.step("matched to a PERMNO, link window respected", len(ev))
    print(f"{n_raw:,} raw  ->  {len(ev):,} matched", flush=True)

    db = wrds.Connection(wrds_username=args.user) if args.user else wrds.Connection()

    years = sorted(ev["session_date"].dt.year.unique())
    years = [y for y in years if 2006 <= y <= 2025]

    cal = trading_calendar(db, years)
    rolled = roll_to_session(ev["session_date"], cal)
    moved = int((rolled.notna() & (rolled != ev["session_date"])).sum())
    off_calendar = int(rolled.isna().sum())
    ev["session_date_raw"] = ev["session_date"]
    ev["session_date"] = rolled
    ev = ev.dropna(subset=["session_date"])
    print(f"rolled to the next session: {moved:,} events "
          f"({100 * moved / max(len(rolled), 1):.2f}%), "
          f"{off_calendar:,} past the end of the calendar", flush=True)
    funnel.step("release session rolled onto the trading calendar", len(ev))

    years = sorted(ev["session_date"].dt.year.unique())
    years = [y for y in years if 2006 <= y <= 2025]

    bp = nyse_breakpoints(db, years)
    print(f"NYSE breakpoints for {len(bp)} years", flush=True)

    bridge = ibes_crsp_bridge(db)
    bridge["edate"] = bridge["edate"].fillna(pd.Timestamp("2099-12-31"))
    print(f"IBES-CRSP bridge: {len(bridge):,} rows", flush=True)

    kept = []
    #exact per-stage accounting, summed across years, so that the bottom of
    #the funnel equals the number of rows actually written.
    cnt = {"in": 0, "cal": 0, "px": 0, "price": 0, "cap": 0, "cov": 0}

    for year in years:
        sub = ev[ev["session_date"].dt.year == year].copy()
        permnos = sorted(sub["permno"].dropna().unique().tolist())
        if not permnos:
            continue

        cnt["in"] += len(sub)
        px = crsp_for_year(db, year, permnos)
        if px.empty:
            continue

        #t-2 on the market's trading calendar, not on calendar days and not on the subset of dates this year's permnos happened to trade.
        #A calendar built from the holdings themselves shifts whenever the
        #holdings change, which would make the reference date depend on the sample rather than on the market.
        sub["i"] = cal.searchsorted(pd.DatetimeIndex(sub["session_date"]),
                                    side="left")
        sub = sub[(sub["i"] >= 2) & (sub["i"] < len(cal))]
        cnt["cal"] += len(sub)
        sub["ref_date"] = cal[sub["i"].to_numpy() - 2]

        sub = sub.merge(px, left_on=["permno", "ref_date"],
                        right_on=["permno", "date"], how="inner")
        cnt["px"] += len(sub)

        sub = sub[sub["prc"] > args.min_price]
        cnt["price"] += len(sub)

        cut = bp.loc[bp["year"] == year, "nyse_p20"]
        cut = float(cut.iloc[0]) if len(cut) else 0.0
        sub = sub[sub["mktcap"] > cut]
        cnt["cap"] += len(sub)

        if sub.empty:
            continue

        #permno -> IBES ticker, honouring the bridge's validity window
        need = sub[["accession", "permno", "session_date"]].merge(
            bridge, on="permno", how="inner")
        need = need[(need.session_date >= need.sdate)
                    & (need.session_date <= need.edate)]
        tickers = sorted(need["ticker"].dropna().unique().tolist())
        if not tickers:
            continue

        cov = analyst_estimates(db, year, tickers)
        if cov.empty:
            continue
        c = need.merge(cov, on="ticker", how="left")
        c = c[(c.anndats < c.session_date)
              & (c.anndats >= c.session_date - pd.Timedelta(days=90))]
        #distinct people over the whole window, not the sum of the daily counts, so a revision does not buy a second analyst.
        agg = c.groupby("accession")["analys"].nunique().rename("n_analysts")

        sub = sub.merge(agg, on="accession", how="left")
        sub["n_analysts"] = sub["n_analysts"].fillna(0).astype(int)
        sub = sub[sub["n_analysts"] >= args.min_analysts]
        cnt["cov"] += len(sub)

        kept.append(sub)
        print(f"  {year}: {len(sub):,}", flush=True)

    out = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()

    funnel.step("inside the t-2 window of the trading calendar", cnt["cal"])
    funnel.step("has a CRSP price at t-2", cnt["px"])
    funnel.step(f"price at t-2 above ${args.min_price:g}", cnt["price"])
    funnel.step("market cap above the NYSE 20th percentile", cnt["cap"])
    funnel.step(f"at least {args.min_analysts} analysts in the prior 90 days",
                cnt["cov"])

    keep_cols = ["accession", "cik", "permno", "gvkey", "filing_date",
                 "session_date", "session_date_raw", "when", "timing_flag",
                 "prc", "mktcap", "n_analysts"]
    if not out.empty:
        out[keep_cols].sort_values("session_date").to_csv(args.out_sample,
                                                          index=False)

    f = funnel.frame()
    f.to_csv(args.out_funnel, index=False)
    print()
    print(f.to_string(index=False))
    print()
    print(f"sample -> {args.out_sample}  ({len(out):,} rows)")
    print(f"funnel -> {args.out_funnel}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing wrds_match_and_funnel.py


In [ ]:
%%writefile fetch_releases.py
#we download earnings release texts for the 117,926 matched filings.
#we first read each filing's index to identify and download only the relevant exhibit,
#reducing data transfer from about 190 GB to roughly one-seventh of that amount.
#we use parallel workers with a shared SEC rate limit of 10 requests per second.
#results are saved in yearly ZIP files, and interrupted downloads can be resumed.

#python fetch_releases.py --events matched_sample.csv --out-zip-dir releases

from __future__ import annotations

import argparse
import csv
import html
import os
import queue
import re
import sys
import threading
import time
import urllib.error
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor

USER_AGENT = "MIT 15.S08 project mbonnetderoovere@gmail.com"
RATE = 9.0          #below the SEC's ceiling of 10 requests a second
WORKERS = 8
MIN_WORDS = 500

ROW_RE = re.compile(r"<tr[^>]*>(.*?)</tr>", re.S | re.I)
CELL_RE = re.compile(r"<td[^>]*>(.*?)</td>", re.S | re.I)
HREF_RE = re.compile(r'href="([^"]+)"', re.I)
SCRIPT_RE = re.compile(r"<(script|style)\b.*?</\1>", re.S | re.I)
TABLE_RE = re.compile(r"<table\b.*?</table>", re.S | re.I)
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"[ \t\r\f\v]+")
NL_RE = re.compile(r"\n{3,}")


class RateLimiter:

    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch(url: str, limiter: RateLimiter, retries: int = 4) -> bytes | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def _strip(fragment: str) -> str:
    return html.unescape(TAG_RE.sub(" ", fragment)).replace("\xa0", " ")


def to_text(raw: str) -> tuple[str, int]:

    body = SCRIPT_RE.sub(" ", raw)

    dropped = 0

    def decide(m: re.Match) -> str:
        nonlocal dropped
        inner = _strip(m.group(0))
        letters = sum(c.isalpha() for c in inner)
        digits = sum(c.isdigit() for c in inner)
        if digits > 0.30 * max(letters + digits, 1):
            dropped += 1
            return " "
        return " " + inner + " "

    body = TABLE_RE.sub(decide, body)
    body = _strip(body)
    body = NL_RE.sub("\n\n", WS_RE.sub(" ", body))
    return body.strip(), dropped


def looks_like_prose(text: str) -> bool:
    words = text.split()
    if len(words) < MIN_WORDS or text.count(". ") < 10:
        return False
    return sum(c.isdigit() for c in text) / max(len(text), 1) < 0.25


def rank(doc_type: str) -> int:
    t = doc_type.upper()
    if t == "EX-99.1":
        return 0
    if t == "EX-99.2":
        return 1
    if t.startswith("EX-99"):
        return 2
    if t == "8-K":
        return 3
    return 9


def parse_index(body: str, base: str) -> list[tuple[str, str]]:
    out = []
    for row in ROW_RE.findall(body):
        cells = CELL_RE.findall(row)
        if len(cells) < 4:
            continue
        doc_type = re.sub(r"<[^>]+>", " ", cells[3]).strip()
        if rank(doc_type) == 9:
            continue
        m = HREF_RE.search(cells[2])
        if not m:
            continue
        href = html.unescape(m.group(1))

        if "ix?doc=" in href:
            href = href.split("ix?doc=", 1)[1]
        url = href if href.startswith("http") else \
            "https://www.sec.gov" + (href if href.startswith("/") else "/" + href)
        out.append((doc_type, url))
    out.sort(key=lambda p: rank(p[0]))
    return out


def one_filing(row: dict, limiter: RateLimiter) -> tuple[dict, str | None]:
    acc = row["accession"]
    cik = str(row["cik"]).lstrip("0")
    base = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{acc.replace('-', '')}/")
    rec = {"accession": acc, "cik": cik,
           "filing_date": row.get("filing_date", ""),
           "source": "", "filename": "", "words": 0, "tables_dropped": 0}

    idx = fetch(base + f"{acc}-index.htm", limiter)
    if idx is None:
        rec["source"] = "index fetch failed"
        return rec, None

    for doc_type, url in parse_index(idx.decode("utf-8", "replace"), base):
        body = fetch(url, limiter)
        if body is None:
            continue
        text, n_tables = to_text(body.decode("utf-8", "replace"))
        if looks_like_prose(text):
            rec["source"] = doc_type if doc_type.upper().startswith("EX-99") \
                else "8-K body"
            rec["filename"] = url.rsplit("/", 1)[-1]
            rec["words"] = len(text.split())
            rec["tables_dropped"] = n_tables
            return rec, text

    rec["source"] = "none"
    return rec, None


def already_done(zip_dir: str) -> set[str]:
    done: set[str] = set()
    if not os.path.isdir(zip_dir):
        return done
    for name in sorted(os.listdir(zip_dir)):
        if not name.endswith(".zip"):
            continue
        try:
            with zipfile.ZipFile(os.path.join(zip_dir, name)) as z:
                done.update(n[:-4] for n in z.namelist() if n.endswith(".txt"))
        except zipfile.BadZipFile:
            print(f"  warning: {name} is corrupt, ignoring", file=sys.stderr)
    return done


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default="matched_sample.csv")
    ap.add_argument("--out-zip-dir", default="releases")
    ap.add_argument("--report", default="extraction_report_full.csv")
    ap.add_argument("--workers", type=int, default=WORKERS)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    with open(args.events, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))

    os.makedirs(args.out_zip_dir, exist_ok=True)
    done = already_done(args.out_zip_dir)
    if done:
        before = len(rows)
        rows = [r for r in rows if r["accession"] not in done]
        print(f"resuming: {before - len(rows):,} stored, {len(rows):,} to go",
              flush=True)
    if args.limit:
        rows = rows[:args.limit]

    limiter = RateLimiter(RATE)
    writer_q: queue.Queue = queue.Queue(maxsize=2000)
    report_rows: list[dict] = []
    report_lock = threading.Lock()

    def writer() -> None:
        zips: dict[str, zipfile.ZipFile] = {}
        while True:
            item = writer_q.get()
            if item is None:
                break
            year, acc, text = item
            if year not in zips:
                zips[year] = zipfile.ZipFile(
                    os.path.join(args.out_zip_dir, f"releases_{year}.zip"),
                    "a", zipfile.ZIP_DEFLATED, compresslevel=6)
            zips[year].writestr(f"{acc}.txt", text)
            writer_q.task_done()
        for z in zips.values():
            z.close()
        writer_q.task_done()

    wt = threading.Thread(target=writer, daemon=True)
    wt.start()

    t0 = time.monotonic()
    counter = {"n": 0}

    def work(row: dict) -> None:
        rec, text = one_filing(row, limiter)
        if text is not None:
            year = (row.get("session_date") or row.get("filing_date") or "")[:4]
            writer_q.put((year or "unknown", rec["accession"], text))
        with report_lock:
            report_rows.append(rec)
            counter["n"] += 1
            n = counter["n"]
        if n % 500 == 0:
            el = time.monotonic() - t0
            rate = n / el
            left = (len(rows) - n) / rate / 3600 if rate else 0
            print(f"  {n:,}/{len(rows):,}  {rate:.1f} filings/s  "
                  f"{left:.1f} h left", flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, rows))

    writer_q.put(None)
    wt.join()

    if not report_rows:
        print("nothing processed")
        return 0

    with open(args.report, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(report_rows[0].keys()))
        w.writeheader()
        w.writerows(report_rows)

    n = len(report_rows)
    counts: dict[str, int] = {}
    for r in report_rows:
        counts[r["source"]] = counts.get(r["source"], 0) + 1
    print(f"\nfilings processed: {n:,} in {(time.monotonic()-t0)/3600:.2f} h")
    for src, c in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {src:<22} {c:>7,}  {100*c/n:5.1f}%")
    ok = sorted(r["words"] for r in report_rows if r["words"])
    if ok:
        print(f"word count: median {ok[len(ok)//2]:,}  "
              f"p10 {ok[len(ok)//10]:,}  p90 {ok[(9*len(ok))//10]:,}")
    print(f"report -> {args.report}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing fetch_releases.py


In [ ]:
%%writefile validate_timing.py
#we compare EDGAR release sessions with I/B/E/S announcement dates and times
#to check whether independent sources agree on the timing of each release.
#we treat ANNTIMS = 00:00:00 as missing and exclude ACTDATS/ACTTIMS, which reflect
#when I/B/E/S recorded the data rather than when the announcement was made.
#if filingDate is later than the acceptance date, we use 6:00 AM ET on filingDate.
#we calculate the agreement rate, count excluded events, and save conflicting
#cases for manual validation against newswire timestamps.

#python validate_timing.py


from __future__ import annotations

import pandas as pd
import wrds

def sessions(dates: pd.Series, times: pd.Series) -> pd.DataFrame:
    #we assign each release to its trading session based on its date and time.
    #after-close releases move to the next day, while missing times remain unknown
    #and cannot confirm a session. We parse ANNTIMS and use vectorized operations
    #to efficiently process the I/B/E/S data.
    d = pd.to_datetime(dates, errors="coerce")
    t = pd.to_datetime(times.astype("string"), format="%H:%M:%S",
                       errors="coerce")
    secs = t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second

    missing = secs.isna() | (secs == 0)       # 00:00:00 is missing, not midnight
    after = ~missing & (secs >= 16 * 3600)
    pre = ~missing & (secs < 9 * 3600 + 1800)

    sess = d.dt.date.astype("string")
    sess = sess.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))

    flag = pd.Series("intraday", index=d.index, dtype="string")
    flag = flag.mask(after, "after_close").mask(pre, "pre_open")                .mask(missing, "no_time").mask(d.isna(), "no_date")
    return pd.DataFrame({"ibes_session": sess.where(d.notna()),
                         "ibes_flag": flag})


def main() -> int:
    ev = pd.read_csv("matched_sample.csv",
                     usecols=["accession", "permno", "session_date", "when"],
                     dtype={"accession": str, "permno": "Int64",
                            "session_date": str, "when": str})
    idn = pd.read_csv("identity.csv", usecols=["accession"], dtype=str)
    ev = ev[ev.accession.isin(set(idn.accession))]
    print(f"{len(ev):,} matched events", flush=True)

    db = wrds.Connection(wrds_username="mahaut22")

    print("crsp.dsf_v2 coverage:",
          db.raw_sql("select max(dlycaldt) as d from crsp.dsf_v2")
            .d.iloc[0], flush=True)

    bridge = db.raw_sql("""
        select distinct ticker as ibtic, permno, sdate, edate
        from wrdsapps.ibcrsphist
        where score <= 3
    """, date_cols=["sdate", "edate"])

    #only the quarterly EPS actual, and only ANNDATS/ANNTIMS. The actuals file carries one row per firm-period, so this is a small table.
    act = db.raw_sql("""
        select ticker as ibtic, pends, anndats, anntims
        from ibes.actu_epsus
        where measure = 'EPS' and pdicity = 'QTR'
          and anndats between '2005-12-01' and '2026-01-31'
    """, date_cols=["pends", "anndats"])
    db.close()
    print(f"ibes actuals {len(act):,}, bridge rows {len(bridge):,}",
          flush=True)

    act["ibtic"] = act.ibtic.str.strip()
    bridge["ibtic"] = bridge.ibtic.str.strip()
    act["permno"] = pd.NA

    a = act.merge(bridge, on="ibtic", how="inner", suffixes=("", "_b"))
    a = a[(a.anndats >= a.sdate) & (a.anndats <= a.edate)]
    a["permno"] = a.permno_b.astype("Int64")

    a = pd.concat([a.reset_index(drop=True),
                   sessions(a.anndats, a.anntims).reset_index(drop=True)],
                  axis=1)

    ev["_d"] = pd.to_datetime(ev.session_date, errors="coerce")
    #an I/B/E/S announcement is matched to the EDGAR event nearest in time,
    #within three calendar days. A wider window would start pairing adjacent quarters and manufacture agreement.
    ev = ev.sort_values("_d")
    a = a.sort_values("anndats")
    m = pd.merge_asof(ev, a[["permno", "anndats", "anntims", "ibes_session",
                             "ibes_flag"]],
                      left_on="_d", right_on="anndats", by="permno",
                      direction="nearest", tolerance=pd.Timedelta("3D"))

    has = m.ibes_session.notna()
    agree = has & (m.ibes_session == m.session_date)
    timed = has & (m.ibes_flag != "no_time")
    agree_timed = timed & (m.ibes_session == m.session_date)

    n = len(m)
    print(f"\nI/B/E/S announcement found within 3 days: {has.sum():,} "
          f"({100*has.mean():.1f}%)")
    print(f"  of those, a usable ANNTIMS:              {timed.sum():,} "
          f"({100*timed.sum()/max(has.sum(),1):.1f}%)")
    print(f"\nsame session, all matched:   {agree.sum():,} "
          f"({100*agree.sum()/max(has.sum(),1):.1f}%)")
    print(f"same session, timed only:    {agree_timed.sum():,} "
          f"({100*agree_timed.sum()/max(timed.sum(),1):.1f}%)")

    # The two-source rule the feedback asks for, costed before it is adopted.
    keep = agree
    print(f"\nif an event is kept only when two sources agree:")
    print(f"  kept    {keep.sum():,} ({100*keep.mean():.1f}% of {n:,})")
    print(f"  dropped {n - keep.sum():,} "
          f"({100*(1-keep.mean()):.1f}%), of which")
    print(f"    no I/B/E/S match   {(~has).sum():,}")
    print(f"    match, different session {(has & ~agree).sum():,}")

    diff = (m.loc[has & ~agree, "_d"]
            - m.loc[has & ~agree, "anndats"]).dt.days
    if len(diff):
        print(f"\nwhen they disagree, EDGAR session minus I/B/E/S date:")
        print(diff.value_counts().head(8).sort_index().to_string())

    m["agrees"] = agree
    cols = ["accession", "permno", "session_date", "when", "anndats",
            "anntims", "ibes_session", "ibes_flag", "agrees"]
    m[cols].to_csv("timing_validation.csv", index=False)
    con = m.loc[has & ~agree, cols].head(50)
    con.to_csv("timing_conflicts_50.csv", index=False)
    print("\n-> timing_validation.csv\n-> timing_conflicts_50.csv "
          "(for the hand check against the wire)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing validate_timing.py


In [ ]:
%%writefile wire_times.py
#we use RavenPack Dow Jones timestamps as the primary release time, with EDGAR
#acceptance as a fallback when no wire timestamp is available.
#we convert UTC timestamps to New York time and keep the earliest story per firm
#and day to exclude follow-ups. We match entities to PERMNO using CUSIP and
#validity windows to avoid incorrect matches after mergers.

#python wire_times.py --years 2006-2025

from __future__ import annotations

import argparse

import pandas as pd
import wrds

EARNINGS_LIKE = "earnings%%"
MIN_RELEVANCE = 100


def wire_for_year(db: wrds.Connection, year: int) -> pd.DataFrame:
    """Earliest Dow Jones earnings story per entity and UTC day."""
    sql = f"""
        select rp_entity_id,
               min(timestamp_utc) as wire_utc
        from ravenpack_dj.rpa_djpr_equities_{year}
        where relevance >= {MIN_RELEVANCE}
          and category like '{EARNINGS_LIKE}'
          and news_type = 'PRESS-RELEASE'
        group by rp_entity_id, rpa_date_utc
    """
    try:
        return db.raw_sql(sql)
    except Exception as e:
        #some vintages lack news_type. Fall back rather than lose the year,
        #and say so, because the looser filter admits wire stories that are not the press release itself.
        print(f"  {year}: retrying without news_type ({str(e).splitlines()[0][:60]})",
              flush=True)
        return db.raw_sql(sql.replace(
            "and news_type = 'PRESS-RELEASE'", ""))


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--years", default="2006-2025")
    ap.add_argument("--out", default="wire_times.csv")
    args = ap.parse_args()
    lo, hi = (int(x) for x in args.years.split("-"))

    db = wrds.Connection(wrds_username="mahaut22")

    maps = db.raw_sql("""
        select rp_entity_id, cusip, range_start, range_end
        from ravenpack_common.wrds_rpa_all_mappings
        where cusip is not null
    """, date_cols=["range_start", "range_end"])
    maps["cusip8"] = maps.cusip.str.strip().str[:8]
    print(f"entity to cusip rows: {len(maps):,}", flush=True)

    #CRSP historical CUSIP, so a firm that changed CUSIP is matched on the one it carried at the time rather than its last one.
    names = db.raw_sql("""
        select permno, cusip as cusip8, namedt, nameenddt
        from crsp.stocknames_v2
        where cusip is not null
    """, date_cols=["namedt", "nameenddt"])
    names["cusip8"] = names.cusip8.str.strip().str[:8]
    print(f"crsp cusip rows: {len(names):,}", flush=True)

    frames = []
    for y in range(lo, hi + 1):
        w = wire_for_year(db, y)
        w["wire_utc"] = pd.to_datetime(w.wire_utc, errors="coerce", utc=True)
        w = w.dropna(subset=["wire_utc"])
        frames.append(w)
        print(f"  {y}: {len(w):,} entity-days", flush=True)
    db.close()

    wire = pd.concat(frames, ignore_index=True)
    wire["wire_et"] = wire.wire_utc.dt.tz_convert("America/New_York")
    wire["wire_date"] = wire.wire_et.dt.date.astype("string")

    m = wire.merge(maps[["rp_entity_id", "cusip8", "range_start", "range_end"]],
                   on="rp_entity_id", how="inner")
    d = m.wire_et.dt.tz_localize(None)
    m = m[(d >= m.range_start) & (d <= m.range_end.fillna(pd.Timestamp("2100-01-01")))]

    m = m.merge(names, on="cusip8", how="inner")
    d = m.wire_et.dt.tz_localize(None)
    m = m[(d >= m.namedt) & (d <= m.nameenddt)]
    m["permno"] = m.permno.astype("Int64")

    out = (m.groupby(["permno", "wire_date"], as_index=False)
             .wire_et.min()
             .rename(columns={"wire_et": "wire_time_et"}))

    out["wire_time_et"] = out.wire_time_et.dt.strftime("%Y-%m-%d %H:%M:%S%z")
    out.to_csv(args.out, index=False)

    print(f"\n{len(out):,} firm-days with a Dow Jones wire time")
    print(f"distinct permnos: {out.permno.nunique():,}")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing wire_times.py


In [ ]:
%%writefile three_source_timing.py
#we compare trading sessions from EDGAR, I/B/E/S, and Dow Jones, keeping events
#only when at least two sources agree. We use wire timestamps when available
#and EDGAR as a fallback, without selecting the earliest timestamp.
#we report agreement patterns, count excluded events, and save conflicts for manual validation.

#python three_source_timing.py

from __future__ import annotations

import argparse

import pandas as pd

CLOSE_S = 16 * 3600
OPEN_S = 9 * 3600 + 1800


OFFSET_RE = r"[+-]\d{2}:?\d{2}$|Z$"


def parse_et(s: pd.Series) -> pd.Series:
    #we convert timestamps to New York time, handling both older files without UTC
    #offsets and newer files with offsets. We check the format before parsing to
    #avoid timezone errors and look-ahead bias, including daylight saving changes.
    txt = s.astype("string").str.strip()
    has_offset = txt.dropna().str.contains(OFFSET_RE, regex=True, na=False)
    if len(has_offset) and has_offset.any():
        t = pd.to_datetime(txt, errors="coerce", utc=True)
        return t.dt.tz_convert("America/New_York").dt.tz_localize(None)
    return pd.to_datetime(txt, errors="coerce")


def to_et(ts: pd.Series) -> pd.Series:
    #we convert timezone-aware timestamps to New York time and assume naive timestamps
    #are already in New York time. This ensures correct trading session assignment
    #and avoids classifying after-close releases as intraday.
    t = pd.to_datetime(ts, errors="coerce")
    if getattr(t.dtype, "tz", None) is not None:
        t = t.dt.tz_convert("America/New_York").dt.tz_localize(None)
    return t


def session(dates: pd.Series, secs: pd.Series) -> pd.Series:
    #we assign releases to sessions based on their date and time, moving after-close
    #releases to the next calendar day. Missing times result in missing sessions
    #rather than estimated ones.
    d = to_et(dates)
    missing = secs.isna() | (secs == 0)
    after = ~missing & (secs >= CLOSE_S)
    s = d.dt.date.astype("string")
    s = s.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))
    return s.where(d.notna() & ~missing)


def secs_of(ts: pd.Series) -> pd.Series:
    t = to_et(ts)
    return t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--keep-all", action="store_true",
                    help="write every event, corroborated or not. "
                         "The default keeps only the events two "
                         "sources place in the same session.")
    args = ap.parse_args()

    t = pd.read_csv("timing_validation.csv", dtype={"accession": str})
    w = pd.read_csv("wire_times.csv")

    #we match wire stories using the exact release timestamp rather than the session
    #date to avoid incorrect matches. We use release_et when available, or fall back to acceptance_et for older event tables.

    head = pd.read_csv("event_table.csv", nrows=0).columns
    col = "release_et" if "release_et" in head else "acceptance_et"
    if col != "release_et":
        print("event_table.csv has no release_et column, so the 06:00 rule "
              "for filings disseminated the next morning is not applied "
              "here. Rebuild it with the current build_event_table.py.")
    ev = pd.read_csv("event_table.csv", dtype={"accession": str},
                     usecols=["accession", col]).rename(
                         columns={col: "release_et"})
    ev = ev.drop_duplicates("accession")
    t = t.merge(ev, on="accession", how="left", validate="many_to_one")
    print(f"{len(t):,} events, {len(w):,} firm-days with a wire time, "
          f"{t.release_et.notna().sum():,} with a release timestamp")

    t["permno"] = pd.to_numeric(t.permno, errors="coerce").astype("Int64")
    w["permno"] = pd.to_numeric(w.permno, errors="coerce").astype("Int64")

    #each event is matched to the closest wire story for the same firm within three days,
    #using the same tolerance as I/B/E/S. Both timestamps are parsed as UTC using their
    #stored timezone offsets to avoid incorrect matches and timing errors.

    w["wire_dt"] = parse_et(w.wire_time_et)
    t["edgar_dt"] = parse_et(t.release_et)
    t = t.dropna(subset=["edgar_dt"])
    t = t.sort_values("edgar_dt")
    w = w.sort_values("wire_dt").dropna(subset=["wire_dt"])

    m = pd.merge_asof(t, w[["permno", "wire_dt"]], left_on="edgar_dt",
                      right_on="wire_dt", by="permno", direction="nearest",
                      tolerance=pd.Timedelta("3D"))

    m["wire_session"] = session(m.wire_dt, secs_of(m.wire_dt))
    m["ibes_session_c"] = m.ibes_session
    m["edgar_session"] = m.session_date

    has_w = m.wire_session.notna()
    has_i = m.ibes_session_c.notna()
    n = len(m)

    print(f"\ncoverage")
    print(f"  EDGAR session            {n:>8,} (100.0%)")
    print(f"  I/B/E/S session          {has_i.sum():>8,} "
          f"({100*has_i.mean():.1f}%)")
    print(f"  Dow Jones wire session   {has_w.sum():>8,} "
          f"({100*has_w.mean():.1f}%)")
    print(f"  all three                {(has_i & has_w).sum():>8,} "
          f"({100*(has_i & has_w).mean():.1f}%)")

    ei = has_i & (m.edgar_session == m.ibes_session_c)
    ew = has_w & (m.edgar_session == m.wire_session)
    iw = has_i & has_w & (m.ibes_session_c == m.wire_session)

    print(f"\npairwise agreement, among events where both sources exist")
    print(f"  EDGAR with I/B/E/S       {ei.sum():>8,} of {has_i.sum():,} "
          f"({100*ei.sum()/max(has_i.sum(),1):.1f}%)")
    print(f"  EDGAR with the wire      {ew.sum():>8,} of {has_w.sum():,} "
          f"({100*ew.sum()/max(has_w.sum(),1):.1f}%)")
    print(f"  I/B/E/S with the wire    {iw.sum():>8,} of "
          f"{(has_i & has_w).sum():,} "
          f"({100*iw.sum()/max((has_i & has_w).sum(),1):.1f}%)")

    two = (ei.astype(int) + ew.astype(int) + iw.astype(int)) >= 1
    m["agree_two"] = two
    print(f"\nkeeping an event only when at least two sources agree")
    print(f"  kept     {two.sum():>8,} ({100*two.mean():.1f}%)")
    print(f"  dropped  {n - two.sum():>8,} ({100*(1-two.mean()):.1f}%)")

    only_edgar = ~has_i & ~has_w
    print(f"    of the dropped, with no second source at all: "
          f"{(only_edgar).sum():,}")
    print(f"    of the dropped, with a second source that disagrees: "
          f"{(n - two.sum() - only_edgar.sum()):,}")

    #where the wire speaks, it is the release time. This is the column the return windows should key on, with EDGAR as the stated fallback.

    m["session_final"] = m.wire_session.where(has_w, m.edgar_session)
    m["session_source"] = pd.Series("edgar", index=m.index).where(
        ~has_w, "wire")
    moved = (m.session_final != m.edgar_session).sum()
    print(f"\nadopting the wire where it exists moves {moved:,} events "
          f"({100*moved/n:.1f}%) to another session")

    print("\nhow far the wire moves the session, in sessions")
    delta = (pd.to_datetime(m.session_final, errors="coerce")
             - pd.to_datetime(m.edgar_session, errors="coerce")).dt.days
    print(delta[delta != 0].value_counts().head(8).sort_index().to_string())

    cols = ["accession", "permno", "edgar_session", "when", "ibes_session_c",
            "ibes_flag", "wire_dt", "wire_session", "session_final",
            "session_source", "agree_two"]


    m[cols].to_csv("three_source_timing_all.csv", index=False)
    kept = m if args.keep_all else m[two]
    kept[cols].to_csv("three_source_timing.csv", index=False)
    print(f"\nthree_source_timing.csv holds {len(kept):,} events"
          + ("" if args.keep_all else
             f", {n - len(kept):,} dropped for want of a second source"))


    conflict = m[has_w & has_i & ~(ei & ew & iw)]
    take = conflict.sample(min(50, len(conflict)), random_state=20261104)
    hand = take[cols].copy()
    hand["edgar_url"] = [
        "https://www.sec.gov/Archives/edgar/data/"
        + f"{a.replace('-', '')}/{a}-index.htm" for a in hand.accession]
    for c in ["session_you_read", "source_you_trust", "note"]:
        hand[c] = ""
    hand.to_csv("timing_handcheck_50.csv", index=False,
                encoding="utf-8-sig")
    print("-> three_source_timing_all.csv\n-> three_source_timing.csv"
          "\n-> timing_handcheck_50.csv")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing three_source_timing.py


In [ ]:
%%writefile check_access.py
#WRDS access is checked for I/B/E/S Guidance, SEC linking tables, RavenPack,
#and CRSP data tiers. A one-row query is used for each dataset to confirm
#that the data is accessible and not empty, rather than relying on schema visibility.

#python check_access.py


from __future__ import annotations

import wrds

CHECKS = [
    ("CRSP daily, CIZ", "select max(dlycaldt) as v from crsp.dsf_v2"),
    ("CRSP daily, legacy", "select max(date) as v from crsp.dsf"),
    ("I/B/E/S Guidance", "select count(*) as v from ibes.det_guidance "
                         "where anndats >= '2024-01-01'"),
    ("I/B/E/S Guidance cols", "select * from ibes.det_guidance limit 1"),
    ("I/B/E/S actuals, unadj", "select max(anndats) as v "
                               "from ibes.actu_epsus"),
    ("I/B/E/S detail, unadj", "select max(anndats) as v "
                              "from ibes.detu_epsus"),
    ("IBES-CRSP bridge", "select count(*) as v from wrdsapps.ibcrsphist"),
    ("CCM link", "select count(*) as v from crsp.ccmxpf_lnkhist"),
    ("SEC analytics suite", "select count(*) as v "
                            "from wrdssec.wciklink_gvkey"),
    ("RavenPack equities", "select count(*) as v from ravenpack.rpa_djpr_equities "
                           "limit 1"),
    ("RavenPack, any", "select count(*) as v "
                       "from information_schema.tables "
                       "where table_schema like 'ravenpack%'"),
    ("Loughran-McDonald", "select count(*) as v "
                          "from information_schema.tables "
                          "where table_schema like '%loughran%' "
                          "or table_name like '%loughran%'"),
]


def main() -> int:
    db = wrds.Connection(wrds_username="mahaut22")
    for name, sql in CHECKS:
        try:
            df = db.raw_sql(sql)
            if "v" in df.columns:
                print(f"  OK      {name:<24} {df.v.iloc[0]}")
            else:
                print(f"  OK      {name:<24} {len(df.columns)} columns: "
                      f"{', '.join(list(df.columns)[:12])}")
        except Exception as e:
            msg = str(e).split("\n")[0][:90]
            print(f"  DENIED  {name:<24} {msg}")
    db.close()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing check_access.py


In [ ]:
#acquisition pass
import subprocess, sys

PIPELINE = [
    'funnel_counts.py',
    'build_event_table.py',
    'fetch_acceptance.py',
    'wrds_match_and_funnel.py',
    'fetch_releases.py',
    'validate_timing.py',
    'wire_times.py',
    'three_source_timing.py',
    'check_access.py'
]

if RUN_FROM_SCRATCH:
    for script in PIPELINE:
        print("=" * 60, script, flush=True)
        if script == "build_event_table.py":
            subprocess.run([
                sys.executable, script,
                "--zip", ZIP,
                "--out", os.path.join(DATA, "event_table.csv"),
                "--acceptance",
                "/content/drive/MyDrive/S08 NLP/files csv/acceptance_times.csv"
            ], check=True)
        else:
            subprocess.run([sys.executable, script], check=True)

else:
    for script in PIPELINE:
        print("written, not run:", script)

written, not run: funnel_counts.py
written, not run: build_event_table.py
written, not run: fetch_acceptance.py
written, not run: wrds_match_and_funnel.py
written, not run: fetch_releases.py
written, not run: validate_timing.py
written, not run: wire_times.py
written, not run: three_source_timing.py
written, not run: check_access.py


In [ ]:
ok, bad = 0, []


def check(label, got, want, pct=None):
    global ok
    hit = (round(got, 4) == round(want, 4))
    if hit:
        ok += 1
    else:
        bad.append(f"{label}: got {got}, report says {want}")
    tail = f"  ({pct:.1f}%)" if pct is not None else ""
    print(f"  {'ok ' if hit else 'NO '}{label:<52}{got:>12,}{tail}")

In [ ]:
print("1. raw counts")
rc = pd.read_csv(f"{DATA}/raw_counts.csv")
#the file carries its own otal row; the yearly rows are the data.
total = rc[rc.year == "TOTAL"].iloc[0]
rc = rc[rc.year != "TOTAL"].astype({"year": int})
check("TOTAL row agrees with the sum of the years", int(rc.item_2_02_filings.sum()), int(total.item_2_02_filings))
check("Item 2.02 filings, 2006-2025", int(rc.item_2_02_filings.sum()), 383153)
check("all 8-K filings", int(rc.all_8k_filings.sum()), 1549567)
check("fewest distinct filers in a year", int(rc.distinct_filers_with_2_02.min()), 4277)
check("most distinct filers in a year", int(rc.distinct_filers_with_2_02.max()), 6003)
check("years covered", int(rc.year.nunique()), 20)
first, last = rc.set_index("year").item_2_02_filings.loc[[2006, 2025]]
check("fall in Item 2.02 count, 2006 to 2025, in percent", round(100 * (1 - last / first)), 27)

1. raw counts
  ok TOTAL row agrees with the sum of the years               383,153
  ok Item 2.02 filings, 2006-2025                             383,153
  ok all 8-K filings                                        1,549,567
  ok fewest distinct filers in a year                           4,277
  ok most distinct filers in a year                             6,003
  ok years covered                                                 20
  ok fall in Item 2.02 count, 2006 to 2025, in percent             27


In [ ]:
print("\n2. match attrition")
fn = pd.read_csv(f"{DATA}/attrition_funnel.csv")
ms = pd.read_csv(f"{DATA}/matched_sample.csv")
check("events after the last stage", int(fn.remaining.iloc[-1]), 117926)
check("matched sample rows", len(ms), 117926)
rem = fn.remaining.to_numpy()
check("stages where removed equals the drop in remaining",
      int(sum(int(fn.removed.iloc[i]) == rem[i - 1] - rem[i]
              for i in range(1, len(fn)))), len(fn) - 1)
check("largest single removal, the size cut", int(fn.removed.max()), 79505)
check("median market capitalisation, $bn",
      round(ms.mktcap.median() / 1e6, 1), 3.4)
check("median number of analysts", int(ms.n_analysts.median()), 9)
check("events in 2025, lost with the legacy CRSP file",
      int((pd.to_datetime(ms.session_date).dt.year == 2025).sum()), 5791)


2. match attrition
  NO events after the last stage                              117,836
  ok matched sample rows                                      117,926
  ok stages where removed equals the drop in remaining              6
  NO largest single removal, the size cut                      79,685
  ok median market capitalisation, $bn                            3.4
  ok median number of analysts                                      9
  ok events in 2025, lost with the legacy CRSP file             5,791


In [ ]:
print("\n3. release text")
ex = pd.read_csv(f"{DATA}/extraction_report.csv")
src = ex.source.fillna("")
got = src.ne("none") & src.ne("index fetch failed")
check("events with a release recovered", int(got.sum()), 116983,
      100 * got.mean())
v1 = src.str.startswith("EX-99.1")
other = src.str.startswith("EX-99") & ~v1
check("EX-99.1 and its variants", int(v1.sum()), 103852, 100 * v1.mean())
check("other EX-99 exhibits", int(other.sum()), 11269, 100 * other.mean())
body = src.eq("8-K body")
check("the 8-K body itself", int(body.sum()), 1862, 100 * body.mean())
check("no prose document found", int(src.eq("none").sum()), 941,
      100 * src.eq("none").mean())
check("index page unreachable", int(src.eq("index fetch failed").sum()), 2)
check("share lost by taking EX-99.1 alone, in percent",
      round(100 * (got.sum() - v1.sum()) / got.sum(), 1), 11.2)
w = ex.loc[got, "words"]
check("median release, words", int(w.median()), 3157)
check("tenth percentile, words", int(w.quantile(0.10)), 1437)
check("ninetieth percentile, words", round(w.quantile(0.90)), 5960)
check("median tables dropped", int(ex.loc[got, "tables_dropped"].median()), 3)


3. release text
  ok events with a release recovered                          116,983  (99.2%)
  ok EX-99.1 and its variants                                 103,852  (88.1%)
  ok other EX-99 exhibits                                      11,269  (9.6%)
  ok the 8-K body itself                                        1,862  (1.6%)
  ok no prose document found                                      941  (0.8%)
  ok index page unreachable                                         2
  ok share lost by taking EX-99.1 alone, in percent              11.2
  ok median release, words                                      3,157
  ok tenth percentile, words                                    1,437
  ok ninetieth percentile, words                                5,960
  ok median tables dropped                                          3


In [ ]:
print("\n4. release timing")
CLOSE_S, OPEN_S = 16 * 3600, 9 * 3600 + 1800


def session(dates, secs):
    d = pd.to_datetime(dates, errors="coerce")
    missing = secs.isna() | (secs == 0)
    after = ~missing & (secs >= CLOSE_S)
    s = d.dt.date.astype("string")
    s = s.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))
    return s.where(d.notna() & ~missing)


def secs_of(ts):
    t = pd.to_datetime(ts, errors="coerce")
    return t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second


t = pd.read_csv(f"{DATA}/timing_validation.csv")
wi = pd.read_csv(f"{DATA}/wire_times.csv")
n = len(t)
check("firm-days with a Dow Jones wire time", len(wi), 444915)
check("distinct securities on the wire file", int(wi.permno.nunique()), 8853)

has_i_all = t.ibes_flag.notna()
check("events with an I/B/E/S announcement", int(has_i_all.sum()), 106418,
      100 * has_i_all.mean())
check("EDGAR and I/B/E/S on the same session", int(t.agrees.sum()), 88448,
      100 * t.agrees.sum() / has_i_all.sum())
check("kept if two sources must agree, in percent",
      round(100 * t.agrees.mean(), 1), 75.0)
check("dropped by that rule", int(n - t.agrees.sum()), 29478)
check("  of those, no I/B/E/S match", int((~has_i_all).sum()), 11508)
dis = (~t.agrees) & has_i_all
check("  of those, matched to a different session", int(dis.sum()), 17970)

d = t[dis].copy()
d["lag"] = (pd.to_datetime(d.session_date)
            - pd.to_datetime(d.anndats)).dt.days
same = d.lag == 0
check("same date, I/B/E/S after the close, EDGAR in session",
      int((same & d.when.eq("intraday")).sum()), 12809)
check("same date, I/B/E/S after the close, EDGAR pre-open",
      int((same & d.when.eq("pre_open")).sum()), 154)
check("same date, both after the close, different session",
      int((same & d.when.eq("after_close")).sum()), 27)
check("same date, differing only on the side of the close",
      int(same.sum()), 12990)
check("EDGAR session one to three days after the I/B/E/S date",
      int(d.lag.between(1, 3).sum()), 4706)
check("EDGAR session before the I/B/E/S date", int((d.lag < 0).sum()), 274)

t["permno"] = pd.to_numeric(t.permno, errors="coerce").astype("Int64")
wi["permno"] = pd.to_numeric(wi.permno, errors="coerce").astype("Int64")
wi["wire_dt"] = pd.to_datetime(wi.wire_time_et, errors="coerce")
t["edgar_dt"] = pd.to_datetime(t.session_date, errors="coerce")
m = pd.merge_asof(t.sort_values("edgar_dt"),
                  wi.sort_values("wire_dt").dropna(subset=["wire_dt"])
                    [["permno", "wire_dt"]],
                  left_on="edgar_dt", right_on="wire_dt", by="permno",
                  direction="nearest", tolerance=pd.Timedelta("3D"))
m["wire_session"] = session(m.wire_dt, secs_of(m.wire_dt))
has_w, has_i = m.wire_session.notna(), m.ibes_session.notna()
check("events given a wire time", int(has_w.sum()), 90437, 100 * has_w.mean())

ei = has_i & (m.session_date == m.ibes_session)
ew = has_w & (m.session_date == m.wire_session)
iw = has_i & has_w & (m.ibes_session == m.wire_session)
check("I/B/E/S agrees with the wire", int(iw.sum()), 86374,
      100 * iw.sum() / (has_i & has_w).sum())
check("  out of events where both exist", int((has_i & has_w).sum()), 87647)
check("EDGAR agrees with the wire", int(ew.sum()), 75775,
      100 * ew.sum() / has_w.sum())
check("EDGAR agrees with I/B/E/S", int(ei.sum()), 88448,
      100 * ei.sum() / has_i.sum())

two = (ei.astype(int) + ew.astype(int) + iw.astype(int)) >= 1
check("retained when two of three sources agree", int(two.sum()), 104259,
      100 * two.mean())
check("dropped by that rule", int(n - two.sum()), 13667)
only_e = ~has_i & ~has_w
check("  of those, with no second source", int(only_e.sum()), 8718)
check("  of those, with a second source that disagrees",
      int(n - two.sum() - only_e.sum()), 4949)

m["session_final"] = m.wire_session.where(has_w, m.session_date)
moved = m.session_final != m.session_date
check("events the wire moves to another session", int(moved.sum()), 14662,
      100 * moved.mean())
delta = (pd.to_datetime(m.session_final, errors="coerce")
         - pd.to_datetime(m.session_date, errors="coerce")).dt.days
check("  EDGAR one to three sessions too early", int((delta > 0).sum()), 10526)
check("  EDGAR one to three sessions too late", int((delta < 0).sum()), 4136)


4. release timing
  ok firm-days with a Dow Jones wire time                     444,915
  ok distinct securities on the wire file                       8,853
  ok events with an I/B/E/S announcement                      106,418  (90.2%)
  ok EDGAR and I/B/E/S on the same session                     88,448  (83.1%)
  ok kept if two sources must agree, in percent                  75.0
  ok dropped by that rule                                      29,478
  ok   of those, no I/B/E/S match                              11,508
  ok   of those, matched to a different session                17,970
  ok same date, I/B/E/S after the close, EDGAR in session      12,809
  ok same date, I/B/E/S after the close, EDGAR pre-open           154
  ok same date, both after the close, different session            27
  ok same date, differing only on the side of the close        12,990
  ok EDGAR session one to three days after the I/B/E/S date       4,706
  ok EDGAR session before the I/B/E/S date         

In [ ]:
chk = pd.read_csv(f"{DATA}/acceptance_header_check.csv")
got = chk.dropna(subset=["delta_hours"])
same = (got.delta_hours == 0)

print("EDGAR acceptance time: the submissions JSON against the filing header")
print(f"  filings checked                 {len(got):>6,}")
print(f"  JSON agrees with the header     {same.sum():>6,}  "
      f"{100 * same.mean():5.1f}%")
print(f"  JSON disagrees                  {(~same).sum():>6,}  "
      f"{100 * (~same).mean():5.1f}%")
print("\n  what the field actually holds, decided filing by filing")
for name, n in chk.regime.value_counts().items():
    print(f"    {n:>5,}  {name}")
print("""
    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter""")
print("  Three regimes mixed inside the same year, so no rule recovers the")
print("  truth from the JSON alone. fetch_acceptance.py reads the header.")

print("\nStill to rerun, in this order, once the headers are fetched:")
for s in ["fetch_acceptance.py", "build_event_table.py",
          "wrds_match_and_funnel.py", "validate_timing.py",
          "three_source_timing.py"]:
    print("   ", s)
print("\nThe release texts do not need refetching: the extraction does not")
print("depend on the timestamp. The CRSP windows do.")

EDGAR acceptance time: the submissions JSON against the filing header
  filings checked                    300
  JSON agrees with the header         99   33.0%
  JSON disagrees                     201   67.0%

  what the field actually holds, decided filing by filing
      128  Eastern labelled Z
       99  true UTC
       73  converted to UTC twice

    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter
  Three regimes mixed inside the same year, so no rule recovers the
  truth from the JSON alone. fetch_acceptance.py reads the header.

Still to rerun, in this order, o

In [ ]:
ts = pd.read_csv(f"{DATA}/three_source_timing.csv",
                 dtype={"accession": str})
conflict = ts[ts.wire_session.notna() & ts.ibes_session_c.notna()
              & ((ts.edgar_session != ts.wire_session)
                 | (ts.edgar_session != ts.ibes_session_c))]

hand = conflict.sample(min(50, len(conflict)), random_state=20261104).copy()
hand["edgar_url"] = [
    "https://www.sec.gov/Archives/edgar/data/"
    + f"{a.replace('-', '')}/{a}-index.htm" for a in hand.accession]
for c in ["session_you_read", "source_you_trust", "note"]:
    hand[c] = ""

path = os.path.join(MANUAL, "timing_handcheck_50.csv")
hand.to_csv(path, index=False, encoding="utf-8-sig")
print(f"{len(conflict):,} events where the sources disagree, "
      f"{len(hand)} drawn")
print("->", path)
print()
print("For each row: open the filing on EDGAR, read the acceptance time in")
print("the header, and say which session the release really belongs to.")
print("`source_you_trust` is edgar, ibes or wire. The point is not the")
print("fifty verdicts, it is whether one source is wrong in one direction.")

14,317 events where the sources disagree, 50 drawn
-> /content/drive/MyDrive/S08 NLP/results/2_EDGAR text/Manual validation/timing_handcheck_50.csv

For each row: open the filing on EDGAR, read the acceptance time in
the header, and say which session the release really belongs to.
`source_you_trust` is edgar, ibes or wire. The point is not the
fifty verdicts, it is whether one source is wrong in one direction.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fn = pd.read_csv(f"{DATA}/attrition_funnel.csv")
rc = pd.read_csv(f"{DATA}/raw_counts.csv")
ms = pd.read_csv(f"{DATA}/matched_sample.csv",
                 usecols=["session_date"])
ts = pd.read_csv(f"{DATA}/three_source_timing.csv",
                 usecols=["edgar_session", "wire_session", "ibes_session_c"])

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

a = ax[0][0]
a.barh(fn.step[::-1], fn.remaining[::-1], color="#4a5568")
a.set_title("Attrition funnel")
a.set_xlabel("filings remaining")
for i, n in enumerate(fn.remaining[::-1]):
    a.text(n, i, f" {n:,}", va="center", fontsize=8)
a.tick_params(axis="y", labelsize=8)

a = ax[0][1]
y = rc[rc.year != "TOTAL"].copy()
y["year"] = y.year.astype(int)
a.plot(y.year, y.item_2_02_filings, marker="o", color="#2c5282",
       label="Item 2.02 filings")
ev_per_year = (pd.to_datetime(ms.session_date, errors="coerce")
                 .dt.year.value_counts().sort_index())
a.plot(ev_per_year.index, ev_per_year.values, marker="o", color="#9b2c2c",
       label="in the final sample")
a.set_title("Filings a year, before and after the filters")
a.legend(frameon=False)

a = ax[1][0]
#each rate is over the events where both sources speak. Dividing by every event instead would mix "the two disagree" with "one is silent", which are different facts and move the number by ten points.
def agree(a, b):
    both = ts[a].notna() & ts[b].notna()
    return (ts.loc[both, a] == ts.loc[both, b]).mean(), int(both.sum())


pairs = {f"EDGAR = I/B/E/S\nn={agree('edgar_session', 'ibes_session_c')[1]:,}":
         agree("edgar_session", "ibes_session_c")[0],
         f"EDGAR = wire\nn={agree('edgar_session', 'wire_session')[1]:,}":
         agree("edgar_session", "wire_session")[0],
         f"I/B/E/S = wire\nn={agree('ibes_session_c', 'wire_session')[1]:,}":
         agree("ibes_session_c", "wire_session")[0]}
a.bar(list(pairs), [100 * v for v in pairs.values()], color="#4a5568")
a.set_ylim(0, 100)
a.set_ylabel("% of events in the same session")
a.set_title("Do two sources put the release in the same session?")
for i, v in enumerate(pairs.values()):
    a.text(i, 100 * v, f"{100 * v:.1f}%", ha="center", va="bottom",
           fontsize=9)

a = ax[1][1]
delta = (pd.to_datetime(ts.wire_session, errors="coerce")
         - pd.to_datetime(ts.edgar_session, errors="coerce")).dt.days
d = delta[(delta.notna()) & (delta != 0) & (delta.abs() <= 5)]
a.bar(d.value_counts().sort_index().index,
      d.value_counts().sort_index().values, color="#9b2c2c")
a.set_title("How far the wire moves the session")
a.set_xlabel("sessions, wire minus EDGAR")

for row in ax:
    for a in row:
        a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
#access

QUERIES = {
    "crsp.dsf_v2": "select max(dlycaldt) from crsp.dsf_v2",
    "crsp.dsf": "select max(date) from crsp.dsf",
    "ibes.det_guidance": "select count(*) from ibes.det_guidance "
                         "where anndats >= '2024-01-01'",
    "ibes.actu_epsus": "select max(anndats) from ibes.actu_epsus",
    "ibes.detu_epsus": "select max(anndats) from ibes.detu_epsus",
    "wrdsapps.ibcrsphist": "select count(*) from wrdsapps.ibcrsphist "
                           "where score <= 3",
    "crsp.ccmxpf_lnkhist": "select count(*) from crsp.ccmxpf_lnkhist",
    "wrdssec.wciklink_gvkey": "select count(*) from wrdssec.wciklink_gvkey",
}

if RUN_FROM_SCRATCH:
    !pip install -q wrds
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USER)
    for table, q in QUERIES.items():
        try:
            print(f"{table:<26}{db.raw_sql(q).iloc[0, 0]}")
        except Exception as exc:
            print(f"{table:<26}refused: {type(exc).__name__}")
    db.close()
else:
    for table, q in QUERIES.items():
        print(f"{table:<26}{q}")

crsp.dsf_v2               select max(dlycaldt) from crsp.dsf_v2
crsp.dsf                  select max(date) from crsp.dsf
ibes.det_guidance         select count(*) from ibes.det_guidance where anndats >= '2024-01-01'
ibes.actu_epsus           select max(anndats) from ibes.actu_epsus
ibes.detu_epsus           select max(anndats) from ibes.detu_epsus
wrdsapps.ibcrsphist       select count(*) from wrdsapps.ibcrsphist where score <= 3
crsp.ccmxpf_lnkhist       select count(*) from crsp.ccmxpf_lnkhist
wrdssec.wciklink_gvkey    select count(*) from wrdssec.wciklink_gvkey
